In [ ]:
import pandas as pd
import numpy as np

# 1. Load your data
df = pd.read_csv('hotel_bookings.csv')  # Your 119k rows
df_hotels = pd.read_csv('hotels.csv')   # Your 20 hotel IDs

# 2. Assign a repeating number (1-10) to each booking row
# We group by 'hotel' so that 'Resort' gets 1-10 and 'City' gets 1-10 independently
df['hotel_num'] = df.groupby('hotel').cumcount() % 10 + 1

# 3. Clean the Hotel IDs in the seed dataframe
# We extract the digits from 'RES01' to get '1' so they match our hotel_num
df_hotels['id_num'] = df_hotels['hotel_id'].str.extract('(\d+)').astype(int)

# 4. Perform the "Composite Join"
# We match on BOTH the text (Resort/City) AND the assigned number (1-10)
df_final = pd.merge(
    df, 
    df_hotels, 
    left_on=['hotel', 'hotel_num'], 
    right_on=['hotel_type', 'id_num'], 
    how='left'
)

# 5. Clean up helper columns
df_final = df_final.drop(columns=['hotel_num', 'id_num', 'hotel_type'])

# Check for nulls now
print(f"Nulls in hotel_id: {df_final['hotel_id'].isnull().sum()}")
print(df_final[['hotel', 'hotel_id', 'hotel_name']].head(15))


Nulls in hotel_id: 0
           hotel hotel_id              hotel_name
0   Resort Hotel    RES01      Azure Sands Resort
1   Resort Hotel    RES02       Alpine Peak Lodge
2   Resort Hotel    RES03     Emerald Bay Retreat
3   Resort Hotel    RES04     Sahara Mirage Oasis
4   Resort Hotel    RES05       Cancun Sun Palace
5   Resort Hotel    RES06     Sydney Harbour View
6   Resort Hotel    RES07        Bali Zen Gardens
7   Resort Hotel    RES08       Cape Town Coastal
8   Resort Hotel    RES09    Maldives Blue Lagoon
9   Resort Hotel    RES10  Santorini White Cliffs
10  Resort Hotel    RES01      Azure Sands Resort
11  Resort Hotel    RES02       Alpine Peak Lodge
12  Resort Hotel    RES03     Emerald Bay Retreat
13  Resort Hotel    RES04     Sahara Mirage Oasis
14  Resort Hotel    RES05       Cancun Sun Palace


In [4]:
df_final.to_csv('final_bookings.csv', index=False)

In [2]:
print('\n'.join(df.columns))

hotel
is_canceled
lead_time
arrival_date_year
arrival_date_month
arrival_date_week_number
arrival_date_day_of_month
stays_in_weekend_nights
stays_in_week_nights
adults
children
babies
meal
country
market_segment
distribution_channel
is_repeated_guest
previous_cancellations
previous_bookings_not_canceled
reserved_room_type
assigned_room_type
booking_changes
deposit_type
agent
company
days_in_waiting_list
customer_type
adr
required_car_parking_spaces
total_of_special_requests
reservation_status
reservation_status_date


In [8]:
# Drop only columns that exist in the dataframe
columns_to_drop = ['meal', 'booking_changes', 'agent', 'days_in_waiting_list', 'required_car_parking_spaces','company']
columns_to_drop = [col for col in columns_to_drop if col in df.columns]
df = df.drop(columns_to_drop, axis=1)
df.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,deposit_type,customer_type,adr,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,0,0,C,C,No Deposit,Transient,0.0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,0,0,C,C,No Deposit,Transient,0.0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,0,0,A,C,No Deposit,Transient,75.0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,0,0,A,A,No Deposit,Transient,75.0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,0,0,A,A,No Deposit,Transient,98.0,1,Check-Out,2015-07-03


In [9]:
df['arrival_date'] = pd.to_datetime(df['arrival_date_year'].astype(str) + '-' + df['arrival_date_month'] + '-' + df['arrival_date_day_of_month'].astype(str))
df = df.drop(['arrival_date_year', 'arrival_date_month', 'arrival_date_day_of_month', 'arrival_date_week_number'], axis=1)
df.head()

,hotel,is_canceled,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,country,market_segment,...,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,deposit_type,customer_type,adr,total_of_special_requests,reservation_status,reservation_status_date,arrival_date
0,Resort Hotel,0,342,0,0,2,0.0,0,PRT,Direct,...,0,C,C,No Deposit,Transient,0.0,0,Check-Out,2015-07-01,2015-07-01
1,Resort Hotel,0,737,0,0,2,0.0,0,PRT,Direct,...,0,C,C,No Deposit,Transient,0.0,0,Check-Out,2015-07-01,2015-07-01
2,Resort Hotel,0,7,0,1,1,0.0,0,GBR,Direct,...,0,A,C,No Deposit,Transient,75.0,0,Check-Out,2015-07-02,2015-07-01
3,Resort Hotel,0,13,0,1,1,0.0,0,GBR,Corporate,...,0,A,A,No Deposit,Transient,75.0,0,Check-Out,2015-07-02,2015-07-01
4,Resort Hotel,0,14,0,2,2,0.0,0,GBR,Online TA,...,0,A,A,No Deposit,Transient,98.0,1,Check-Out,2015-07-03,2015-07-01


In [10]:
df.describe(include='all')

,hotel,is_canceled,lead_time,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,country,market_segment,...,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,deposit_type,customer_type,adr,total_of_special_requests,reservation_status,reservation_status_date,arrival_date
count,119390,119390.000000,119390.000000,119390.000000,119390.000000,119390.000000,119386.000000,119390.000000,118902,119390,...,119390.000000,119390,119390,119390,119390,119390.000000,119390.000000,119390,119390,119390
unique,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,177,8,...,NaN,10,12,3,4,NaN,NaN,3,926,NaN
top,City Hotel,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PRT,Online TA,...,NaN,A,A,No Deposit,Transient,NaN,NaN,Check-Out,2015-10-21,NaN
freq,79330,NaN,NaN,NaN,NaN,NaN,NaN,NaN,48590,56477,...,NaN,85994,74053,104641,89613,NaN,NaN,75166,1461,NaN
mean,NaN,0.370416,104.011416,0.927599,2.500302,1.856403,0.103890,0.007949,NaN,NaN,...,0.137097,NaN,NaN,NaN,NaN,101.831122,0.571363,NaN,NaN,2016-08-28 16:39:45.727447808
min,NaN,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,-6.380000,0.000000,NaN,NaN,2015-07-01 00:00:00
25%,NaN,0.000000,18.000000,0.000000,1.000000,2.000000,0.000000,0.000000,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,69.290000,0.000000,NaN,NaN,2016-03-13 00:00:00
50%,NaN,0.000000,69.000000,1.000000,2.000000,2.000000,0.000000,0.000000,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,94.575000,0.000000,NaN,NaN,2016-09-06 00:00:00
75%,NaN,1.000000,160.000000,2.000000,3.000000,2.000000,0.000000,0.000000,NaN,NaN,...,0.000000,NaN,NaN,NaN,NaN,126.000000,1.000000,NaN,NaN,2017-03-18 00:00:00
max,NaN,1.000000,737.000000,19.000000,50.000000,55.000000,10.000000,10.000000,NaN,NaN,...,72.000000,NaN,NaN,NaN,NaN,5400.000000,5.000000,NaN,NaN,2017-08-31 00:00:00


Cleaning Tasks with dbt
1. Remove all Nulls
2. Add,year, month into a new column arrival_date. Change arrival date to normal date no numbers and remove all non date like figures
3. Convert 0,1s to booleans
4. Properly rename columns
5. Flag family booking into a new column
6. total nights = stays in weekend + stays in week
7. 

Things to add to the dataset
1. Synthetic hotel IDs to distinguish
2. Booking revenue

Data Quality
1. Daily rate is never negative
2. Arrival date and hotel is never null
